In [1]:
import sys
!{sys.executable} -m pip install cfgrib eccodes

In [2]:
import xarray as xr
print(xr.backends.list_engines())

{'netcdf4': <NetCDF4BackendEntrypoint>
  Open netCDF (.nc, .nc4 and .cdf) and most HDF5 files using netCDF4 in Xarray
  Learn more at https://docs.xarray.dev/en/stable/generated/xarray.backends.NetCDF4BackendEntrypoint.html, 'scipy': <ScipyBackendEntrypoint>
  Open netCDF files (.nc, .nc4, .cdf and .gz) using scipy in Xarray
  Learn more at https://docs.xarray.dev/en/stable/generated/xarray.backends.ScipyBackendEntrypoint.html, 'cfgrib': <CfGribBackend>
  Open GRIB files (.grib, .grib2, .grb and .grb2) in Xarray
  Learn more at https://github.com/ecmwf/cfgrib, 'gini': <GiniXarrayBackend>, 'store': <StoreBackendEntrypoint>
  Open AbstractDataStore instances in Xarray
  Learn more at https://docs.xarray.dev/en/stable/generated/xarray.backends.StoreBackendEntrypoint.html}


In [3]:
import cfgrib
import os

data_dir = r'C:\Users\Carolina\Documents\MSc Civil Engineering\Marine renewables\marine renewables\Marine-renewables\data aquisition\data'

files = [f for f in os.listdir(data_dir) if f.endswith('.grib')]
print("GRIB files:", files)

for f in files:
    full_path = os.path.join(data_dir, f)
    print(f"\n--- {f} ---")
    datasets = cfgrib.open_datasets(full_path)
    for i, ds in enumerate(datasets):
        print(f"  Dataset {i}: {list(ds.data_vars)}")

GRIB files: ['311edd7bcd65302389ae099b202aec6e.grib', '4442e4d799f439dfe0452da8fb8de243.grib', '87651c937f5a950f80d1c4291b8e5ed5.grib']

--- 311edd7bcd65302389ae099b202aec6e.grib ---
  Dataset 0: ['u10', 'v10']

--- 4442e4d799f439dfe0452da8fb8de243.grib ---
  Dataset 0: ['u100', 'v100']

--- 87651c937f5a950f80d1c4291b8e5ed5.grib ---
  Dataset 0: ['swh', 'pp1d']


In [5]:
print("u10/v10 length:", len(pt_10m.time.values))
print("u100/v100 length:", len(pt_100m.time.values))
print("wave length:", len(pt_wave.time.values))

print("\n10m time range:", str(pt_10m.time.values[0])[:10], "to", str(pt_10m.time.values[-1])[:10])
print("100m time range:", str(pt_100m.time.values[0])[:10], "to", str(pt_100m.time.values[-1])[:10])
print("wave time range:", str(pt_wave.time.values[0])[:10], "to", str(pt_wave.time.values[-1])[:10])


u10/v10 length: 29224
u100/v100 length: 32144
wave length: 32144

10m time range: 2010-01-01 to 2020-12-31
100m time range: 2010-01-01 to 2020-12-31
wave time range: 2010-01-01 to 2020-12-31


In [6]:
# Check time steps
import numpy as np

times_10m  = pt_10m.time.values
times_100m = pt_100m.time.values
times_wave = pt_wave.time.values

# Check interval between first few timestamps
dt_10m  = (times_10m[1]  - times_10m[0])  / np.timedelta64(1, 'h')
dt_100m = (times_100m[1] - times_100m[0]) / np.timedelta64(1, 'h')
dt_wave = (times_wave[1] - times_wave[0]) / np.timedelta64(1, 'h')

print(f"10m  time step: {dt_10m} hours")
print(f"100m time step: {dt_100m} hours")
print(f"wave time step: {dt_wave} hours")

10m  time step: 3.0 hours
100m time step: 3.0 hours
wave time step: 3.0 hours


In [7]:
import pandas as pd
import numpy as np
import os

data_dir = r'C:\Users\Carolina\Documents\MSc Civil Engineering\Marine renewables\marine renewables\Marine-renewables\data aquisition\data'

# Build individual DataFrames
df_10m = pd.DataFrame({
    'wind_speed_10m_ms': np.sqrt(pt_10m['u10'].values**2 + pt_10m['v10'].values**2),
    'wind_dir_10m_deg':  (270 - np.degrees(np.arctan2(pt_10m['v10'].values, pt_10m['u10'].values))) % 360,
}, index=pd.DatetimeIndex(pt_10m.time.values))

df_100m = pd.DataFrame({
    'wind_speed_100m_ms': np.sqrt(pt_100m['u100'].values**2 + pt_100m['v100'].values**2),
    'wind_dir_100m_deg':  (270 - np.degrees(np.arctan2(pt_100m['v100'].values, pt_100m['u100'].values))) % 360,
}, index=pd.DatetimeIndex(pt_100m.time.values))

df_wave = pd.DataFrame({
    'Hs_m': pt_wave['swh'].values,
    'Tp_s': pt_wave['pp1d'].values,
}, index=pd.DatetimeIndex(pt_wave.time.values))

# Merge on common timestamps only
df_wind = df_10m.join(df_100m, how='inner')  # keeps only matching timestamps
print(f"10m records:   {len(df_10m)}")
print(f"100m records:  {len(df_100m)}")
print(f"Merged wind:   {len(df_wind)}")
print(f"Wave records:  {len(df_wave)}")
print(f"Time step 10m: {(df_10m.index[1]-df_10m.index[0]).seconds/3600}h")
print(f"Time step 100m:{(df_100m.index[1]-df_100m.index[0]).seconds/3600}h")

# Save CSVs
df_wind.to_csv(os.path.join(data_dir, 'wind_data_sines_2010_2020.csv'))
df_wave.to_csv(os.path.join(data_dir, 'wave_data_sines_2010_2020.csv'))

print("\nWind sample:")
print(df_wind.head())
print("\nWave sample:")
print(df_wave.head())
print("\nCSVs saved!")

10m records:   29224
100m records:  32144
Merged wind:   29224
Wave records:  32144
Time step 10m: 3.0h
Time step 100m:3.0h

Wind sample:
                     wind_speed_10m_ms  wind_dir_10m_deg  wind_speed_100m_ms  \
2010-01-01 00:00:00          11.378973        274.830200           13.204857   
2010-01-01 03:00:00          12.063279        277.673767           14.090853   
2010-01-01 06:00:00          12.025712        283.169434           14.135208   
2010-01-01 09:00:00          11.216561        287.584412           13.078920   
2010-01-01 12:00:00           9.727643        286.973114           11.236833   

                     wind_dir_100m_deg  
2010-01-01 00:00:00         274.922241  
2010-01-01 03:00:00         277.744537  
2010-01-01 06:00:00         282.981934  
2010-01-01 09:00:00         287.573883  
2010-01-01 12:00:00         287.262848  

Wave sample:
                         Hs_m       Tp_s
2010-01-01 00:00:00  4.356820  11.297707
2010-01-01 03:00:00  4.161001  11.20639